###  BPVA checkpoint 加载

In [1]:
import os
from pathlib import Path

import torch

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.BPVA.configuration_bpva import BPVAConfig
from lerobot.policies.BPVA.modeling_bpva import BPVAPolicy

CHECKPOINT_DIR = Path(
    "/vla/workspace/my_tbot/outputs/BPVA/SFT-Robotwin/2026-08-13/17-37-26_bpva_train/checkpoints/200000/pretrained_model"
)
DEVICE = "cuda"

for filename in ("config.json", "model.safetensors"):
    checkpoint_file = CHECKPOINT_DIR / filename
    if not checkpoint_file.is_file():
        raise FileNotFoundError(f"checkpoint 文件不存在：{checkpoint_file}")

# 必须由配置基类根据 config.json 中的 type="bpva" 解析具体配置类型。
cfg = PreTrainedConfig.from_pretrained(CHECKPOINT_DIR, local_files_only=True)
if not isinstance(cfg, BPVAConfig):
    raise TypeError(f"checkpoint 配置类型不是 BPVAConfig，而是 {type(cfg).__name__}")
cfg.device = DEVICE
# checkpoint 已包含 ViT 权重；构建模型时不再联网下载 timm 权重。
cfg.bp_vision_pretrained = False
cfg.bp_vision_checkpoint_path = None
# 单次 forward 不计算 DA3 teacher loss，避免加载额外 teacher 权重。
cfg.lambda_3d = 0.0

policy = BPVAPolicy.from_pretrained(
    CHECKPOINT_DIR,
    config=cfg,
    local_files_only=True,
    # 该 checkpoint 使用 tied embedding，保存时会省略共享键；沿用项目默认的非严格加载。
    strict=False,
)
policy.eval()

print("checkpoint:", CHECKPOINT_DIR)
print("policy name:", policy.name)
print("device:", cfg.device, "dtype:", cfg.dtype)
print("num state_dict keys:", len(policy.state_dict()))


DecodingError: The fields `type` are not valid for BPVAConfig

In [ ]:
# 确认 behavior prompt encoder 权重已从 checkpoint 恢复。
bp_keys = [key for key in policy.state_dict() if key.startswith("model.bp_obs_encoder.")]
if not bp_keys:
    raise RuntimeError("checkpoint 中未加载 model.bp_obs_encoder.* 权重")

print("bp_num_chunks:", cfg.bp_num_chunks)
print("bp_action_chunk_size:", cfg.bp_action_chunk_size)
print("num bp_obs_encoder keys:", len(bp_keys))
print("first bp key:", bp_keys[0])

### 按原有方式加载数据集并构造一个 batch

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_PATH = "/vla/workspace/data/robotwin2.0/hanging_mug/aloha-agilex_clean_50"
ACTION_CHUNK_SIZE = cfg.bp_action_chunk_size
FPS = 30

current_delta_timestamps = {
    "action": [i / FPS for i in range(ACTION_CHUNK_SIZE)],
    "observation.images.cam_high": [-0.5, 0.0, 0.5],
    "observation.images.cam_left_wrist": [-0.5, 0.0, 0.5],
    "observation.images.cam_right_wrist": [-0.5, 0.0, 0.5],
}
# prompt_ds：图像只取当前关键帧，action 仍返回完整未来窗口。
prompt_delta_timestamps = {
    "action": [i / FPS for i in range(ACTION_CHUNK_SIZE)],
}
current_ds = LeRobotDataset(DATASET_PATH, delta_timestamps=current_delta_timestamps)
prompt_ds = LeRobotDataset(DATASET_PATH, delta_timestamps=prompt_delta_timestamps)


In [ ]:
from lerobot.datasets.behavior_prompt_dataset import BehaviorPromptConfig, BehaviorPromptLeRobotDataset

# 参数与训练时 factory.py 构造 BehaviorPromptConfig 的方式保持一致。
bp_config = BehaviorPromptConfig(
    prompt_action_chunk_size=cfg.bp_action_chunk_size,
    max_prompt_chunks=None,
    num_chunks=cfg.bp_num_chunks,
    same_episode_policy="avoid",
    seed=0,
    height=cfg.image_resolution[0],
    width=cfg.image_resolution[1],
    max_state_dim=cfg.max_state_dim,
    max_action_dim=cfg.max_action_dim,
    qwen3_vl_processor_path=cfg.qwen3_vl_pretrained_path,
    bp_camera_keys=list(cfg.bp_camera_keys),
    action_mode="delta",
)

bp_ds = BehaviorPromptLeRobotDataset.with_default_transforms(current_ds, prompt_ds, bp_config)
sample = bp_ds[0]
for i, step in enumerate(bp_ds.transform.transforms):
    print(f"data process step: [{i}] {step.__class__.__name__}")
print("BP state:", sample["behavior_prompt"]["state"].shape)
print("BP action:", sample["behavior_prompt"]["action"].shape)
bp_ds

In [ ]:
from torch.utils.data._utils.collate import default_collate

# 单样本 batch 足以验证完整 forward，并尽量降低显存占用。
samples = [bp_ds[0]]
batch = default_collate(samples)

def move_to_device(value, device):
    if isinstance(value, torch.Tensor):
        return value.to(device)
    if isinstance(value, dict):
        return {key: move_to_device(item, device) for key, item in value.items()}
    if isinstance(value, list):
        return [move_to_device(item, device) for item in value]
    if isinstance(value, tuple):
        return tuple(move_to_device(item, device) for item in value)
    return value

batch_for_forward = move_to_device(batch, DEVICE)

In [ ]:
# 使用真实数据执行一次 forward；不反向传播，避免保留额外梯度和显存。
torch.cuda.empty_cache()
policy.eval()
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    loss, loss_dict = policy(batch_for_forward)

print("forward ok")
print("loss:", float(loss.cpu()))
for key, value in loss_dict.items():
    if key.startswith("loss_action_dim"):
        continue
    print(f"{key}: {value}")